# ReCaLL for code — frozen StarCoder2-3B
Standalone fixed experiment. Paper Eq. (2): mean_LL(target | prefix) / mean_LL(target), **higher is member-like**. This is a documented code adaptation: one deterministic same-language 128-token shadow non-member prefix per sample, and an identical target mask in both branches. No sweeps, fitting, label-based direction selection or reference model. See packaged README for the reference implementation's first-token discrepancy and normalization.

Both modes validate the complete saved 1600/200/200/1000 manifests. Smoke selects stratified 80/20/20/30 rows; full uses all saved rows. Prefixes always come from the complete saved shadow non-member pool, so smoke/full plans are compatible. Kaggle GPU inference only; no existing attacks are executed or modified.

In [ ]:
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
UPLOAD_SOURCE = INPUT_ROOT / "datasets/sayonsujitmondal/recall_kaggle_source"
UPLOAD_ZIP = UPLOAD_SOURCE / "recall_kaggle_source.zip"
PROJECT_ROOT = WORK_ROOT / "recall-code-source-v2"
BASELINE_RUN = "/kaggle/input/datasets/sayonsujitmondal/sersem-baseline-1000-results3"
PATH_OVERRIDES = dict(shadow="", validation="", calibration="", heldout="", scores="", config="")
ADAPTIVE_SCORES = ""  # optional; missing/ambiguous adaptive files never block inference
MODE = "smoke"  # "smoke" or "full"
SMOKE_SIZES = dict(shadow=80, validation=20, calibration=20, heldout=30)
RUN_ID = f"recall-code-{MODE}-v1"
OUT = WORK_ROOT / "recall_outputs" / RUN_ID
RESUME_FROM = ""
CACHE_REUSE_FROM = ""
MODEL_CACHE_DIR = ""
MODEL_LOCAL_FILES_ONLY = True
ALLOW_MODEL_DOWNLOAD = True
DATASET_CACHE_DIR = WORK_ROOT / "hf_cache/datasets"
SOURCE_ACCESS = "streaming"
SESSION_BUDGET_MINUTES = 480
DISK_RESERVE_GIB = 3.0
WORK_ROOT.mkdir(parents=True, exist_ok=True)


## Verify source before imports
Upload `recall_kaggle_source.zip` as a Kaggle Dataset (or attach extracted contents), and edit `UPLOAD_SOURCE`. Source initialization and dependency setup follow notebook 03. An old or modified package fails before imports. GPU T4 and Internet are needed for streaming; no full dataset download/preparation is called.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
SOURCE_MANIFEST_SHA256 = 'd2e6f0cc8874cc02d426d4dc24961c01b4897d2f75a02999fb27ad696fb48934'
# Kaggle may normalize Dataset slugs; accept a unique ZIP or extracted package.
if not UPLOAD_SOURCE.exists():
    direct = [INPUT_ROOT / "datasets/sayonsujitmondal/recall-kaggle-source",
              INPUT_ROOT / "recall-kaggle-source", INPUT_ROOT / "recall_kaggle_source"]
    found = [p for p in direct if p.exists()]
    if not found:
        markers = list(INPUT_ROOT.rglob("recall_code_attack/source_manifest.json"))
        archives = list(INPUT_ROOT.rglob("recall_kaggle_source.zip"))
        found = sorted({p.parent.parent for p in markers} | {p.parent for p in archives})
    if len(found) != 1:
        raise FileNotFoundError("Set UPLOAD_SOURCE to the mounted recall_kaggle_source Dataset; candidates: " + str(found))
    UPLOAD_SOURCE = found[0]
    UPLOAD_ZIP = UPLOAD_SOURCE / "recall_kaggle_source.zip"
if not (PROJECT_ROOT / "recall_code_attack/source_manifest.json").is_file():
    if PROJECT_ROOT.exists():
        raise FileExistsError("Incomplete PROJECT_ROOT; select a fresh directory")
    if (UPLOAD_SOURCE / "recall_code_attack/source_manifest.json").is_file():
        source_manifest_path = UPLOAD_SOURCE / "recall_code_attack/source_manifest.json"
        if hashlib.sha256(source_manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
            raise ValueError("Outdated ReCaLL source package")
        source_manifest = json.loads(source_manifest_path.read_text())
        PROJECT_ROOT.mkdir(parents=True)
        for name in list(source_manifest) + ["recall_code_attack/source_manifest.json"]:
            destination = (PROJECT_ROOT / name).resolve()
            if not destination.is_relative_to(PROJECT_ROOT.resolve()):
                raise ValueError("Unsafe source path")
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(UPLOAD_SOURCE / name, destination)
    else:
        with zipfile.ZipFile(UPLOAD_ZIP) as archive:
            for entry in archive.infolist():
                if not (PROJECT_ROOT / entry.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe archive path")
            zipped_manifest = archive.read('recall_code_attack/source_manifest.json')
            if hashlib.sha256(zipped_manifest).hexdigest() != SOURCE_MANIFEST_SHA256:
                raise ValueError('Outdated ReCaLL source ZIP')
            allowed = set(json.loads(zipped_manifest)) | {'recall_code_attack/source_manifest.json'}
            if set(archive.namelist()) != allowed:
                raise ValueError('Unexpected archive files')
            archive.extractall(PROJECT_ROOT)
manifest_path = PROJECT_ROOT / "recall_code_attack/source_manifest.json"
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
    raise ValueError("Source manifest mismatch; use a fresh PROJECT_ROOT and rebuilt package")
source_manifest = json.loads(manifest_path.read_text())
for name, expected in source_manifest.items():
    if hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest() != expected:
        raise ValueError("Source mismatch: " + name)
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "SERSEM-Poisoned-Chalice-Competition-2026"
sys.path.insert(0, str(UPSTREAM / "code"))

In [ ]:
# Set process policy before importing any experiment or model libraries.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# The system dictionary is needed by the pinned pyenchant dependency.
packages = ["libenchant-2-2", "hunspell-en-us"]
status = subprocess.run(["dpkg-query", "-W", "-f=${Status}\n", *packages], capture_output=True, text=True)
if status.returncode != 0 or status.stdout.splitlines().count("install ok installed") != len(packages):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *packages], check=True)
from shared.kaggle_setup import install_dependencies, validate_environment
install_report = install_dependencies(PROJECT_ROOT / "shared/requirements-kaggle.txt", WORK_ROOT)
# A correct fresh-kernel rerun is a no-op. If setup changed loaded modules,
# install_dependencies raises an explicit one-restart instruction.
environment = validate_environment(PROJECT_ROOT / "shared/requirements-kaggle.txt", require_cuda=True)


In [ ]:
# Lightweight synthetic checks only; no pretrained weights or dataset access.
for test_file in ('test_recall.py', 'test_kaggle_setup.py'):
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                    '-p', test_file, '-v'], cwd=PROJECT_ROOT, check=True)


## Validate saved manifests and exact comparison scores
Attach the baseline output Dataset and saved adaptive output Dataset. If the adaptive run is smoke, full evaluation requires its full 1000-row scores. Missing or mismatched IDs fail before inference. Baseline numerical/API versions must match exactly, as in notebook 03. The only smoke sampling seed is the existing baseline seed; **prefix selection always uses 42**. No saved baseline loss is reused.

In [ ]:
import copy
import numpy as np
import pandas as pd
from adaptive_token_attack.data import resolve_paths, load_run, select_partitions, frame_fingerprint
from adaptive_token_attack.features import load_frozen_target
from recall_code_attack.cache import Cache, fingerprint
from recall_code_attack.runner import scoring_identity, prepare_plans, run_scores
from recall_code_attack.evaluation import exact_match, evaluate
from shared.resource_pipeline import Monitor, restore_run
from shared.resource_data import SourceStore
from shared.sersem_baseline import write_json
paths = resolve_paths(BASELINE_RUN, PATH_OVERRIDES, INPUT_ROOT)
original, saved_scores, base_cfg = load_run(paths)
if SMOKE_SIZES != dict(shadow=80, validation=20, calibration=20, heldout=30):
    raise ValueError('This experiment fixes smoke counts to 80/20/20/30')
partitions, selection = select_partitions(original, MODE, SMOKE_SIZES, seed=base_cfg['seed'])
if base_cfg['versions'] != environment['versions']:
    raise ValueError('Restore the saved baseline numerical/API environment before inference/comparison')
adaptive_scores = None
adaptive_path = None
if ADAPTIVE_SCORES:
    adaptive_path = Path(ADAPTIVE_SCORES)
elif len(candidates := sorted(INPUT_ROOT.rglob('heldout_adaptive_scores.parquet'))) == 1:
    adaptive_path = candidates[0]
if adaptive_path is not None:
    adaptive_scores = pd.read_parquet(adaptive_path)
    exact_match(partitions['heldout'], adaptive_scores, ['adaptive_score'])
else:
    print('No unique adaptive score file supplied; inference continues and adaptive paired metrics will be omitted.')
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {'.','..'}:
    raise ValueError('RUN_ID must be a directory name')
if OUT.resolve().is_relative_to(INPUT_ROOT.resolve()):
    raise ValueError('OUT must be writable and separate from inputs')
OUT.mkdir(parents=True,exist_ok=True)
if RESUME_FROM:
    restore_run(RESUME_FROM,OUT)
source_cfg = copy.deepcopy(base_cfg)
source_cfg['resources'].update(data_access=SOURCE_ACCESS,text_cache_gib=.1,
    session_budget_minutes=SESSION_BUDGET_MINUTES,disk_reserve_gib=DISK_RESERVE_GIB)
monitor = Monitor(source_cfg,OUT)
monitor.guard()
print('Selected:',{k:len(v) for k,v in partitions.items()})

## Pin tokenizer, context and execution identity
This metadata/tokenizer setup constructs no target model. Reuses notebook 03's explicit pinned cache preparation (disk checked before any allowed model download). The model is loaded lazily only when a likelihood is missing. Baseline `add_special_tokens=True` and right truncation are retained. The first baseline input token supplies context in both branches and is never scored; all subsequent retained IDs are scored. The prefix is tokenized separately without added special tokens and concatenated as IDs, preserving target boundaries exactly. Context-tail truncation and native sliding attention are recorded. Prefix length cannot be changed.

In [ ]:
from shared.cache_setup import ensure_model_snapshot
from transformers import AutoConfig, AutoTokenizer
cache_dir = MODEL_CACHE_DIR or base_cfg['resources']['model_cache_dir']
ensure_model_snapshot(base_cfg['model_id'],base_cfg['model_revision'],cache_dir,
    ALLOW_MODEL_DOWNLOAD,DISK_RESERVE_GIB,WORK_ROOT,OUT/'model_cache_setup.json')
metadata_options = dict(revision=base_cfg['model_revision'],cache_dir=str(cache_dir),local_files_only=MODEL_LOCAL_FILES_ONLY)
tokenizer = AutoTokenizer.from_pretrained(base_cfg['model_id'],use_fast=True,**metadata_options)
model_config = AutoConfig.from_pretrained(base_cfg['model_id'],**metadata_options)
identity = scoring_identity(base_cfg,tokenizer,model_config,source_manifest)
experiment = dict(schema='recall-code-experiment-v1',mode=MODE,selection=selection,scoring=identity,
    source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    inputs={k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in paths.items()},
    adaptive_scores_sha256=hashlib.sha256(adaptive_path.read_bytes()).hexdigest())
if (OUT/'config.json').exists() and json.loads((OUT/'config.json').read_text()) != experiment:
    raise ValueError('Changed execution identity: choose a new RUN_ID')
write_json(OUT/'config.json',experiment)
write_json(OUT/'source_version_hashes.json',dict(source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    sources=source_manifest,versions=environment,scoring_sha256=fingerprint(identity)))
write_json(OUT/'input_paths.json',dict(**{k:str(v) for k,v in paths.items()},adaptive_scores=str(adaptive_path)))
for role,frame in partitions.items():
    frame.to_parquet(OUT/(role+'_manifest.parquet'),index=False)
cache = Cache(OUT/'recall_cache.sqlite')
if CACHE_REUSE_FROM:
    cache.reuse_from(CACHE_REUSE_FROM)
source_store = SourceStore(source_cfg,OUT,monitor)
plans,pool_hash = prepare_plans(original,partitions,tokenizer,identity,cache,source_store,monitor)
print('Prepared plans:',len(plans),'prefix pool:',pool_hash)

## Frozen inference — bounded likelihoods and durable per-branch caching
Batch size one; 256-token forwards with native KV caching bound logit memory for T4. Model weights use FP16, log-softmax FP32, summation FP64. No hidden states or vocabulary logits are saved. A valid cached branch skips the model call. Each completed branch commits atomically; interruption inside an uncommitted forward requires that branch again. `recall_cache.sqlite` also caches prefix plans, so complete resume skips source iteration. No prefix likelihood enters either score. All 150 selected smoke samples or all 3000 full samples receive a score; only calibration determines the deployed threshold.

In [ ]:
def model_factory():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Missing scores require Kaggle GPU')
    monitor.guard()
    model,loaded_tokenizer = load_frozen_target(base_cfg,cache_dir,MODEL_LOCAL_FILES_ONLY)
    if hashlib.sha256(loaded_tokenizer.backend_tokenizer.to_str().encode()).hexdigest() != identity['tokenizer_sha256']:
        raise ValueError('Inference tokenizer differs from prepared plans')
    return model
write_json(OUT/'run_manifest.json',dict(status='scoring',mode=MODE,selection=selection,
    scoring_sha256=fingerprint(identity),prefix_pool_sha256=pool_hash,source_manifest_sha256=SOURCE_MANIFEST_SHA256))
scored = run_scores(partitions,plans,identity,cache,model_factory,monitor,OUT)
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Scored:',len(scored),'context-truncated:',int((scored.context_truncated_tokens>0).sum()))

## Calibration and exactly paired held-out reporting
Deploy only the calibration threshold. `heldout_diagnostic_roc_only` contains label-dependent ROC summaries and diagnostic thresholds for reporting; they never choose score direction or the deployed threshold. All five methods use exactly matched sample ID, content hash, membership and language (adaptive also matches full saved provenance). No silent inner-join loss. Full must cover 1000/1000 IDs. Smoke is preliminary: only 30/1000 original held-out coverage; low-FPR estimates are coarse. Saved methods keep their original spans; any ReCaLL context truncation is explicit.

In [ ]:
report = evaluate(scored,partitions,saved_scores,adaptive_scores,OUT,MODE)
monitor.guard()
write_json(OUT/'run_manifest.json',dict(status='complete',mode=MODE,selection=selection,
    prefix_pool_sha256=pool_hash,scoring_sha256=fingerprint(identity),source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    input_hashes=experiment['inputs'],adaptive_scores_sha256=experiment['adaptive_scores_sha256'],
    selected_counts={k:len(v) for k,v in partitions.items()},coverage=report['coverage'],
    truncation=dict(context_samples=int((scored.context_truncated_tokens>0).sum()),
        baseline_samples=int((scored.baseline_truncated_tokens>0).sum())),
    artifacts={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir()
               if p.is_file() and p.name in {'config.json','metrics.json','recall_scores.parquet',
                   'matched_comparison.parquet','prefix_manifest.parquet','source_version_hashes.json','threshold.json','recall_cache.sqlite'}}))
cache.close()
print(json.dumps(report,indent=2))

## Persist and resume
Save the **entire OUT directory** using Kaggle Save Version **with outputs** before ending a session. It contains `config.json`, `metrics.json`, `recall_scores.parquet`, `matched_comparison.parquet`, `run_manifest.json`, `prefix_manifest.parquet`, `recall_cache.sqlite`, source/version hashes, selected manifests, threshold and resource logs. The SQLite journal uses DELETE mode with synchronous FULL; completed commits are durable. Preserve journals if copying during a running transaction; stop the cell before persisting.

Same run: attach saved OUT and set `RESUME_FROM` to it, keeping MODE, RUN_ID and scientific configuration unchanged. Smoke → full: use `MODE='full'`, a fresh RUN_ID, `RESUME_FROM=''`, `CACHE_REUSE_FROM` pointing to the saved smoke `recall_cache.sqlite`; attach full adaptive scores. Only compatible plans/likelihood branches are reused. Operational source/cache paths and resource budgets may move. No cache is deleted automatically. Disk/session limits stop at checkpoint boundaries; save outputs and resume.

This reproduces the ReCaLL ratio statistic with documented token-mask and code-prefix adaptations. It is not a claim of equivalent benchmark performance. Non-member provenance relies on the existing SERSEM labels. Fixed n-gram hygiene cannot prove semantic independence of arbitrary programs.